# <center>Transfer Learning<br /> Workshop</center>

![logo_ecole_ing.jpg](attachment:logo_ecole_ing.jpg)
Concepteur : Mellila BOUAM
</div>

# Introduction

The aim of this workshop is to introduce you to [transfer learning](https://fr.wikipedia.org/wiki/Apprentissage_par_transfert), a technique used to exploit the knowledge acquired by a model trained on a large dataset on an application task where the amount of available data is limited.

We begin with an intuitive analysis of the effectiveness of transfer learning. We will then prepare the first part, i.e. the data to be used, of the [image annotations](https://fr.wikipedia.org/wiki/Annotation_automatique_d%27images) (image captioning) workshop in which we will train a model using transfer learning.


## What is transfer learning?

**Transfer Learning** involves taking features learned on one problem, and using them on another similar problem. For example, the features of a model that has learned to recognize cats can be used to initialize a model that will recognize tigers.

Transfer learning is often performed on tasks where the dataset contains too few examples to train a model from scratch.

The typical workflow for transfer learning in the context of deep learning is as follows:

1. Take layers from a pre-trained model.
2. Freeze the weights of these layers.
3. Add trainable layers (whose weights are not frozen) at the end of the model. These will be used to convert features into predictions on a new dataset.
4. Train the new layers on the new dataset.

- What's the point of freezing the layer weights of the pre-trained model (step 2)?
<em>PLEASE COMPLETE</em>


A final optional step, **fine-tuning**, consists in unfreezing all the layers of the model obtained after the previous process and re-training it on the new dataset with a very low learning rate.

- What's the point of the fine-tuning step? Why use a very low learning rate?
<em>PLEASE COMPLETE</em>



# Part 1: Intuition on transfer learning

In this section, we will load the [InceptionV3](https://paperswithcode.com/method/inception-v3) model pre-trained on a classification task on the [ImageNet](https://image-net.org/) dataset. We will then use this model to compute vector representations (called [embeddings](https://developers.google.com/machine-learning/crash-course/embeddings/video-lecture?hl=fr)) of images from the [cats vs dogs](https://www.tensorflow.org/datasets/catalog/cats_vs_dogs) dataset. We will then represent each embedding on the plane and interpret the results.

## Loading the data
We first load the cats vs. dogs dataset using [TFDS](https://www.tensorflow.org/datasets?hl=fr) (TensorFlow Datasets).

Transfer learning works particularly well on small datasets. Complete the code below to display the number of examples in each dataset.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

In [ ]:
# Load a tensorflow dataset

import tensorflow_datasets as tfds

tfds.disable_progress_bar()

train_ds, validation_ds, test_ds = tfds.load(
    #PLEASE COMPLETE
    #"cifar10",
    # Reserve 10% for validation and 10% for test
    split=["train[:40%]", "train[40%:50%]", "train[50%:60%]"],
    as_supervised=True,  # Include labels
)

# Nb of training samples
print("Number of training samples: %d" % tf.data.experimental.cardinality(train_ds))
# Nb of validation samples
print(
    #PLEASE COMPLETE
)
# Nb of test samples
print(
    #PLEASE COMPLETE
)

Let's take a look at the first 9 images in the dataset. Note that class 0 corresponds to "cat" and class 1 to "dog".

In [ ]:
# Displaying a few examples

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 10))
for i, (image, label) in enumerate(train_ds.take(9)):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image)
    plt.title(int(label))
    plt.axis("off")

## Data standardization
The images in the dataset are of different sizes. We therefore need to resize them to match the size expected by the model we're going to use: InceptionV3.

We resize the images to a size of (299x299). To do this, you'll need an image resizing function, which [TensorFLow will provide](https://www.tensorflow.org/api_docs/python/tf/image/resize). Alternatively, if you want to perform the resizing in a single instruction, you can use [lambda expressions](https://www.w3schools.com/python/python_lambda.asp)

In [ ]:
# Resizing images

size = (299, 299)

train_ds = train_ds.map(
    #PLEASE COMPLETE
)
validation_ds =  #PLEASE COMPLETE
test_ds =  #PLEASE COMPLETE


In [ ]:
 # Preprocessing the images to give them as input to InceptionV3

train_ds =  #PLEASE COMPLETE
validation_ds =  #PLEASE COMPLETE
test_ds =  #PLEASE COMPLETE


In [ ]:
batch_size = 32

train_ds = train_ds.cache().batch(batch_size).prefetch(buffer_size=10)
validation_ds =  #PLEASE COMPLETE
test_ds =  #PLEASE COMPLETE


In [ ]:
# Downloading the pre-trained InceptionV3 model with cassification from ImageNet

# If we set weights=None -> we'll get embeddings that don't contain any info
# If we set weights='imagenet' -> we'll get semantically relevant embeddings even if the model hasn't been trained on cats vs dogs or cifar10
image_model = tf.keras.applications.InceptionV3(include_top=False,
                                                weights='imagenet', pooling='avg')

What does the **include_top = false** parameter correspond to?
<em>PLEASE COMPLETE</em>

What does the **weights='imagenet'** parameter correspond to?
<em>PLEASE COMPLETE</em>

What does the **pooling='avg'** parameter correspond to?
<em>PLEASE COMPLETE</em>


We now define the [model we're going to use](https://www.tensorflow.org/api_docs/python/tf/keras/Model):

In [ ]:
# Definition of embedding model using InceptionV3
inputs = keras.Input(shape=(299, 299, 3))
x = image_model(inputs)
model = #PLEASE COMPLETE
model.summary()

We calculate the embeddings of the images in the test dataset. The dataset contains 2326 images and the size of an embedding is 2048 (see the keras.Model documentation above):

In [ ]:
# Calculation of test dataset image embeddings

output = #PLEASE COMPLETE

print(output.shape)

In [ ]:
# Building the list of true classes of the test dataset examples (1 -> dog, 0 -> cat)

labels = []
for _, y in test_ds:
    labels.extend(
        y.numpy().tolist()
    )

To demonstrate the relevance of using an InceptionV3 model pre-trained on ImageNet, we'll visualize the embeddings of the test dataset and see if there's any separation between the dataset's classes.

In order to visualize the embedding of an image, it must be of dimension 2. However, the embeddings calculated are of size 2048. We will therefore use the [t-SNE] algorithm (https://fr.wikipedia.org/wiki/Algorithme_t-SNE) to reduce the dimension from 2048 to 2.

t-SNE is a dimension reduction algorithm used to visualize high-dimensional data. The algorithm attempts to find a configuration that preserves as much proximity as possible between points when moving from high-dimensional space (here 2048) to low-dimensional space (here 2). We will use the implementation of the library [scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html).

In [ ]:
# We apply the tSNE algorithm to embeddings to reduce their size
# and display them (embeddings size 2048 -> 2)
from sklearn.manifold import TSNE
tsne = #PLEASE COMPLETE

In [ ]:
c# We define the color associated with each class (only 2 classes are used in the cats vs dogs case)

colors = {0:'red',
          1:'blue'}

# Plotting the image embeddings calculated by InceptionV3
# Each point corresponds to an image and its coordinates are the image's embedding tsne
plt.figure(figsize=(10, 10))
plt.scatter(*tsne.T, s=1.5, c=[colors[l] for l in labels])

# Part 2: Preparing data for transfer learning on an image captioning task

The aim of this part is to prepare the data, through pre-processing, for training a model on the image captioning task. The image captioning task consists in taking an image as input and giving as output a "sentence" describing what this image contains. This last part will be dealt with in the following Workshop.

We will therefore prepare the data for this next workshop, where we will train a model consisting of an InceptionV3 CNN pre-trained on ImageNet to encode images from the COCO dataset into embeddings, followed by an RNN which will be used to decode the embedding and output the textual description of the image. This training represents a typical example of transfer learning.

One slight modification we can make to the transfer learning workflow described earlier is that, instead of freezing the weights of the pre-trained model (InceptionV3), we can pre-calculate the embeddings of the images in the dataset and save them to disk. This will sacrifice some disk space in exchange for runtime (no need to calculate image embeddings at each epoch) and GPU memory (no need to load the InceptionV3 model during training).

With this modification, on the other hand, it will no longer be possible to perform the fine-tuning phase, which involves completing the training of the InceptionV3 model with a very slight adjustment of the weights (very low learning rate) to optimize the embeddings it builds.

# 1 Downloading data :
First, we need to import the libraries we'll need for this part.

In [ ]:
import tensorflow as tf

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

import collections
import random
import re
import numpy as np
import os
import time
import json
from glob import glob
from PIL import Image
import pickle

You need to download the [MS COCO dataset](https://cocodataset.org/#home). This dataset contains 82,000 annotated images, each with 5 captions. Download the [dataset](http://images.cocodataset.org/zips/train2014.zip), the [corresponding annotations](http://images.cocodataset.org/annotations/annotations_trainval2014.zip), and unzip them into the correct directory. The code below displays the path to the folder containing the data and the annotation file (`annotation_file` and `PATH`).

In [ ]:
# Annotation file path
annotation_folder = os.path.abspath('.')+"/annotations/"
annotation_file = annotation_folder+"captions_train2014.json"

# Path to folder containing images to be annotated
image_folder = '/train2014/'
PATH = os.path.abspath('.') + image_folder

print("annotation file ", annotation_file)
print("PATH path ", PATH)

In [ ]:
with open(annotation_folder+"/captions_val2014.json", 'r') as f:
    annotations = json.load(f)

In [ ]:
with open(annotation_folder+"instances_val2014.json", 'r') as f:
    instances = json.load(f)

We want to select a subset of images to make learning faster. The aim is to take only 35,000 images instead of the 82,000 in the original dataset. To do this, we can create a structure that associates each image with its list of annotations. How could we represent this intermediate data?
<em>PLEASE COMPLETE</em>

In [ ]:
# Reading annotation file
with open(annotation_file, 'r') as f:
    annotations = json.load(f)

# Grouping all annotations with the same identifier.
image_path_to_caption = collections.defaultdict(list)
for val in annotations['annotations']:
    # marking the beginning and end of each annotation
    caption = #PLEASE COMPLETE
    # Image ID is part of the image path
    image_path = PATH + 'COCO_train2014_' + '%012d.jpg' % (val['image_id'])
    # Adding caption associated with image_path
    #PLEASE COMPLETE

In [ ]:
# Taking first images only
image_paths = list(image_path_to_caption.keys())
random.shuffle(image_paths)
train_image_paths = image_paths[:2000]
print(len(train_image_paths))

Now it's time to build the dataset itself. The code producing this dataset is composed of :
<ul>
        <li>The `img_name_vector` list, which contains the names of the image files associated with the annotations. These are the examples in the dataset. </li>
        <li>The `train_captions` list containing the annotations. This is the labels.</li>
</ul>
You can display an example for verification.

In [ ]:
# List of all annotations
train_captions = []
# List of all duplicated image file names (in number of annotations per image)
img_name_vector = []

for image_path in train_image_paths:
    caption_list = image_path_to_caption[image_path]
    # Adding caption_list to train_captions
    #PLEASE COMPLETE
    # Adding duplicate image_path len(caption_list) times
    #PLEASE COMPLETE

In [ ]:
print(len(train_captions), len(img_name_vector))
print(train_captions[0])
Image.open(img_name_vector[0])

# 2 Image pre-processing
What we're going to do next is load the neural network that will handle the vision part of the annotation problem. In fact, the overall architecture of the system will contain two parts:
<ul>
    <li>A CNN whose role will be to summarize and extract the raw and voluminous content of the information carried by the pixels into a compact representation that brings together the essential features to perform the annotation.</li>
    <li>And an RNN which will have as input this compact representation and whose goal is to learn the next word from the first words of the annotation.</li>
</ul>

To create a new model you can use the `tf.keras.Model` function in [tensorflow](https://www.tensorflow.org/api_docs/python/tf/keras/Model). For the CNN part, you won't have to design your own architecture for the neural network, you'll have to use an architecture that already exists and is called [InceptionV3](https://paperswithcode.com/method/inception-v3). You can download the weights of this architecture obtained by pre-training on the [ImageNet](http://www.image-net.org/) dataset (used in classification). This network will therefore be trained to perform image classification, but you will use the last hidden layer of this network (i.e. the penultimate layer, as the last layer corresponds to the ImageNet classes), to later provide input to your RNN. The RNN, as we said earlier, will have to learn to perform annotation based on these image representations.

Since training is generally performed by making several passes over the data, the advantage of precomputing image representations is that you don't have to run the images through the CNN each time, and you avoid the training costs associated with the CNN. The potential disadvantage is that performance is likely to be more limited, as training will only involve the weights of the RNN and not those of the CNN.

In [ ]:
# Downloading the pre-trained InceptionV3 model with cassification from ImageNet
image_model = tf.keras.applications.InceptionV3(include_top=False,
                                                weights='imagenet')
# Creation of a variable that will be the input to the new image pre-processing model
new_input = #PLEASE COMPLETE
# retrieving the last hidden layer containing the image in compact representation
hidden_layer = #PLEASE COMPLETE

# Model that calculates a dense representation of images with InceptionV3
image_features_extract_model = #PLEASE COMPLETE

In [ ]:
def load_image(image_path):
    """
    The load_image function has as input the path of an image and as output a pair
    containing the processed image and its path.
    The load_image function performs the following processing:
        1. Loads the file corresponding to the path image_path
        2. Decodes the image into RGB.
        3. Resize image to size (299, 299).
        4. Normalize image pixels between -1 and 1.
    """
    img = #PLEASE COMPLETE
    img = #PLEASE COMPLETE
    img = #PLEASE COMPLETE
    img = #PLEASE COMPLETE
    return img, image_path

Since you already have a pre-trained CNN, it's best to carry out the training on the representations returned by the CNN, as they are more compact and therefore less costly in terms of memory and time. If your RAM isn't big enough, you'll need to cache them on disk. The last layer of the CNN has the shape `8x8x2048`. You will then store these images as `numpy.array` on disk using the `np.save()` function.

**Note:** Processing in TensorFlow is done by batch (which is easier on the GPU), so you'll need to configure your batch to pass your images to InceptionV3.

In [ ]:
from tqdm import tqdm

# Pre-processing images
# Taking image names
encode_train = sorted(set(img_name_vector))

# Creating an instance of "tf.data.Dataset" based on image names
image_dataset = tf.data.Dataset.from_tensor_slices(encode_train)
# Data split into batches after pre-processing by load_image
image_dataset = image_dataset.map(
  load_image, num_parallel_calls=tf.data.experimental.AUTOTUNE).batch(16)

# Browsing dataset batch by batch for InceptionV3 pre-processing
for img, path in tqdm(image_dataset):
    # InceptionV3 pre-processes the current batch (size (16,8,8,2048))
    batch_features = #PLEASE COMPLETE
    # Resizing batch size (16,8,8,2048) to (16,64,2048)
    batch_features = tf.reshape(batch_features,
                              (batch_features.shape[0], -1, batch_features.shape[3]))
    # Browsing current batch and store path and batch with np.save()
    for bf, p in zip(batch_features, path):
        path_of_feature = p.numpy().decode("utf-8")
        # (image path associated with its new representation , image representation)
        np.save(
            #PLEASE COMPLETE
        )

# 3 Pre-processing annotations

To make annotations we need to be able to process the text automatically by following the steps below:
<ul>
    <li>Build a vocabulary of the words contained in the captions. To do this we need to tokenize (divide) the words in the captions on special characters (spaces, punctuation, special signs, ...).</li>
    <li>This vocabulary will be far too voluminous; we'll have to limit ourselves, for example, to the first 5000 most frequent words. You can replace other words with a special token named "UNK" (unknown).</li>
    <li>The words must also be numbered in the dictionary to form annotations represented in a way that is legible for word processing algorithms.</li>
    <li>The examples (annotations) will be filled in so that they all have the same size (the maximum size of the annotations).</li>
</ul>

In [ ]:
# Finding the maximum size
def calc_max_length(tensor):
    return max(len(t) for t in tensor)

# Selecting the 5000 most frequent words in the vocabulary
top_k = 5000
#The Tokenizer class enables text pre-processing for neural networks
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=top_k,
                                                  oov_token="<unk>",
                                                  filters='!"#$%&()*+.,-/:;=?@[\]^_`{|}~ ')
# Builds a vocabulary based on the train_captions list
tokenizer.fit_on_texts( #PLEASE COMPLETE

# Creating token to fill annotations to equalize length
tokenizer.word_index['<pad>'] = #PLEASE COMPLETE
tokenizer.index_word[0] = #PLEASE COMPLETE

# Creation of vectors (list of integer tokens) from annotations (list of words)
train_seqs = #PLEASE COMPLETE

# Filling each vector up to maximum annotation length
cap_vector = #PLEASE COMPLETE

# Calculates the maximum length used to store attention weights
# It will later be used for display during evaluation
max_length = calc_max_length(train_seqs)

Later (in the next workshop), we'll take up this second part and move on to the training and testing set, the construction of the model we're going to train, and the training and testing phase.